## First, vectorize the CSV data

In [ ]:
'''
The dataset contains transactions made by credit cards in September 2013 by European cardholders.

This dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions. 
The dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions.

Feature 'Class' is the response variable and it takes value 1 in case of fraud and 0 otherwise
'''

In [1]:
import numpy as np
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt

df = pd.read_csv('creditcard.csv')
df.head(10)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
5,2.0,-0.425966,0.960523,1.141109,-0.168252,0.420987,-0.029728,0.476201,0.260314,-0.568671,...,-0.208254,-0.559825,-0.026398,-0.371427,-0.232794,0.105915,0.253844,0.081080,3.67,0
6,4.0,1.229658,0.141004,0.045371,1.202613,0.191881,0.272708,-0.005159,0.081213,0.464960,...,-0.167716,-0.270710,-0.154104,-0.780055,0.750137,-0.257237,0.034507,0.005168,4.99,0
7,7.0,-0.644269,1.417964,1.074380,-0.492199,0.948934,0.428118,1.120631,-3.807864,0.615375,...,1.943465,-1.015455,0.057504,-0.649709,-0.415267,-0.051634,-1.206921,-1.085339,40.80,0
8,7.0,-0.894286,0.286157,-0.113192,-0.271526,2.669599,3.721818,0.370145,0.851084,-0.392048,...,-0.073425,-0.268092,-0.204233,1.011592,0.373205,-0.384157,0.011747,0.142404,93.20,0
9,9.0,-0.338262,1.119593,1.044367,-0.222187,0.499361,-0.246761,0.651583,0.069539,-0.736727,...,-0.246914,-0.633753,-0.120794,-0.385050,-0.069733,0.094199,0.246219,0.083076,3.68,0


In [2]:
df.groupby('Class').size()

Class
0    284315
1       492
dtype: int64

In [3]:
features = df.loc[:, 'Time':'Amount'].to_numpy()
targets  = df.loc[:, 'Class':].to_numpy()

print(features.shape)
print(targets.shape)

(284807, 30)
(284807, 1)


## Prepare a validation set

In [4]:
num_val_samples = int(len(features) * 0.2)
train_features = features[:-num_val_samples]
train_targets = targets[:-num_val_samples]
val_features = features[-num_val_samples:]
val_targets = targets[-num_val_samples:]

print("Number of training samples:", len(train_features))
print("Number of validation samples:", len(val_features))

Number of training samples: 227846
Number of validation samples: 56961


## Analyze class imbalance in the targets

In [5]:
counts = np.bincount(train_targets[:, 0])
print(
    "Number of positive samples in training data: {} ({:.2f}% of total)".format(
        counts[1], 100 * float(counts[1]) / len(train_targets)
    )
)

weight_for_0 = 1.0
weight_for_1 = 100.0 # / counts[1]

print('weight_for_0: ', weight_for_0)
print('weight_for_1: ', weight_for_1)

Number of positive samples in training data: 417 (0.18% of total)
weight_for_0:  1.0
weight_for_1:  100.0


## Normalize the data using training set statistics

In [6]:
mean = np.mean(train_features, axis=0)
train_features -= mean
val_features -= mean

std = np.std(train_features, axis=0)
train_features /= std
val_features /= std

## Build a binary classification model

In [7]:
from tensorflow import keras

model = keras.Sequential([keras.layers.Dense(256, activation="relu", 
                                             input_shape=(train_features.shape[-1],)),
                          keras.layers.Dense(256, activation="relu"),
                          keras.layers.Dropout(0.3),
                          keras.layers.Dense(1, activation="sigmoid")])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 256)               7936      
                                                                 
 dense_1 (Dense)             (None, 256)               65792     
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_2 (Dense)             (None, 1)                 257       
                                                                 
Total params: 73,985
Trainable params: 73,985
Non-trainable params: 0
_________________________________________________________________


## Train the model with `class_weight` argument

In [8]:
import tensorflow_addons as tfa

metrics = [
    keras.metrics.FalseNegatives(name="fn"),
    keras.metrics.FalsePositives(name="fp"),
    keras.metrics.TrueNegatives(name="tn"),
    keras.metrics.TruePositives(name="tp"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
    tfa.metrics.F1Score(num_classes=1, threshold=0.5)]

model.compile(
    optimizer=keras.optimizers.Adam(1e-2), loss="binary_crossentropy", metrics=metrics)
class_weight = {0: weight_for_0, 1: weight_for_1}

model.fit(
    train_features,
    train_targets,
    batch_size=2048,
    epochs=50,
    verbose=2,
    validation_data=(val_features, val_targets),
    class_weight=class_weight)

Epoch 1/50
112/112 - 13s - loss: 0.2048 - fn: 64.0000 - fp: 1905.0000 - tn: 225524.0000 - tp: 353.0000 - precision: 0.1563 - recall: 0.8465 - f1_score: 0.2639 - val_loss: 0.0081 - val_fn: 14.0000 - val_fp: 67.0000 - val_tn: 56819.0000 - val_tp: 61.0000 - val_precision: 0.4766 - val_recall: 0.8133 - val_f1_score: 0.6010 - 13s/epoch - 112ms/step
Epoch 2/50
112/112 - 4s - loss: 0.1023 - fn: 52.0000 - fp: 1306.0000 - tn: 226123.0000 - tp: 365.0000 - precision: 0.2184 - recall: 0.8753 - f1_score: 0.3496 - val_loss: 0.0294 - val_fn: 11.0000 - val_fp: 414.0000 - val_tn: 56472.0000 - val_tp: 64.0000 - val_precision: 0.1339 - val_recall: 0.8533 - val_f1_score: 0.2315 - 4s/epoch - 38ms/step
Epoch 3/50
112/112 - 4s - loss: 0.0938 - fn: 52.0000 - fp: 1296.0000 - tn: 226133.0000 - tp: 365.0000 - precision: 0.2197 - recall: 0.8753 - f1_score: 0.3513 - val_loss: 0.0176 - val_fn: 12.0000 - val_fp: 238.0000 - val_tn: 56648.0000 - val_tp: 63.0000 - val_precision: 0.2093 - val_recall: 0.8400 - val_f1_sco

Epoch 25/50
112/112 - 3s - loss: 0.0225 - fn: 5.0000 - fp: 756.0000 - tn: 226673.0000 - tp: 412.0000 - precision: 0.3527 - recall: 0.9880 - f1_score: 0.5199 - val_loss: 0.0061 - val_fn: 15.0000 - val_fp: 20.0000 - val_tn: 56866.0000 - val_tp: 60.0000 - val_precision: 0.7500 - val_recall: 0.8000 - val_f1_score: 0.7742 - 3s/epoch - 26ms/step
Epoch 26/50
112/112 - 3s - loss: 0.0235 - fn: 10.0000 - fp: 883.0000 - tn: 226546.0000 - tp: 407.0000 - precision: 0.3155 - recall: 0.9760 - f1_score: 0.4769 - val_loss: 0.0059 - val_fn: 14.0000 - val_fp: 52.0000 - val_tn: 56834.0000 - val_tp: 61.0000 - val_precision: 0.5398 - val_recall: 0.8133 - val_f1_score: 0.6489 - 3s/epoch - 25ms/step
Epoch 27/50
112/112 - 3s - loss: 0.0228 - fn: 4.0000 - fp: 743.0000 - tn: 226686.0000 - tp: 413.0000 - precision: 0.3573 - recall: 0.9904 - f1_score: 0.5251 - val_loss: 0.0121 - val_fn: 12.0000 - val_fp: 129.0000 - val_tn: 56757.0000 - val_tp: 63.0000 - val_precision: 0.3281 - val_recall: 0.8400 - val_f1_score: 0.

Epoch 49/50
112/112 - 3s - loss: 0.0182 - fn: 2.0000 - fp: 477.0000 - tn: 226952.0000 - tp: 415.0000 - precision: 0.4652 - recall: 0.9952 - f1_score: 0.6341 - val_loss: 0.0121 - val_fn: 11.0000 - val_fp: 95.0000 - val_tn: 56791.0000 - val_tp: 64.0000 - val_precision: 0.4025 - val_recall: 0.8533 - val_f1_score: 0.5470 - 3s/epoch - 27ms/step
Epoch 50/50
112/112 - 3s - loss: 0.0128 - fn: 4.0000 - fp: 543.0000 - tn: 226886.0000 - tp: 413.0000 - precision: 0.4320 - recall: 0.9904 - f1_score: 0.6016 - val_loss: 0.0112 - val_fn: 11.0000 - val_fp: 77.0000 - val_tn: 56809.0000 - val_tp: 64.0000 - val_precision: 0.4539 - val_recall: 0.8533 - val_f1_score: 0.5926 - 3s/epoch - 27ms/step
